# 2. Configuration

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order. Run `01_introduction.ipynb` first if you haven't.

`nuclearff` reads two different kinds of configuration, and it's worth telling them apart before going further:

- `NuclearffConfig` -- *your* settings: where data lives on disk, and the projection model's knobs (recency weights, age curve, simulation settings). You write this once per project.
- `LeagueConfig` -- *the league's* settings: scoring rules and roster shape, derived from a real Sleeper league. You get one of these per league, and it's derived from Sleeper rather than hand-written.

Mirrors the [Configuration](https://nolmacdonald.github.io/nuclearff/tutorial/02_configuration.html) docs page.

## Project configuration

`default_config()` returns a `NuclearffConfig` populated with documented defaults, ready to write to disk. This notebook writes everything under `./demo` rather than a real project's data tree, the same scratch-directory convention every chapter after this one reuses.

In [1]:
from nuclearff.config import default_config, dump_config, load_config

cfg = default_config()
path = dump_config(cfg, "./demo/configs/nuclearff.yaml")
path

PosixPath('demo/configs/nuclearff.yaml')

In [2]:
cfg = load_config(path, root="./demo")
cfg.paths.ensure()

for directory in cfg.paths.all_dirs():
    print(directory.is_dir(), directory)

True /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/raw
True /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/processed
True /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/cache
True /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/manifests
True /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/data/artifacts
True /Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/configs/leagues


`paths.root` anchors every one of those directories -- `cache_dir` is where the Sleeper client caches the player map and where the shared DuckDB database lives (`04_capturing_a_league.ipynb` onward); `artifacts_dir` is where every figure and report in this tutorial gets written; `leagues_dir` is where a league's derived `LeagueConfig` (below) is stored.

Every field on `ModelConfig` and its nested settings is documented -- the recency weights and weight blocks drive `07_projections.ipynb`, the age curve and simulation settings drive `11_simulation.ipynb`.

In [3]:
cfg.model

ModelConfig(name='wr_default', version='0.1.0', position='WR', seasons=(2023, 2024, 2025), recency=RecencyWeights(weights=(0.5, 0.3, 0.2)), weights=WeightBlocks(volume=0.5, efficiency=0.25, expected_td=0.1, situation=0.15), thresholds=SampleThresholds(min_games=8, min_routes=200, min_targets=50), age_curve=AgeCurveConfig(peak_age=26, plateau_end=28, decline_per_year=0.025, min_factor=0.85), simulation=SimulationConfig(n_simulations=10000, games=17, floor_percentile=0.1, ceiling_percentile=0.9), baseline='vols', flex_wr_rate=0.5)

> **Why this matters:** it's tempting to skip straight to fetching a league and treat this config as boilerplate. It isn't -- every number you'll see printed from Chapter 7 onward (a recency-weighted target share, an age-curve factor, a simulated floor/ceiling) traces back to a field on this exact object. If a later chapter's output ever looks off, `cfg.model` here is the first place to check, not the code that consumes it.

## League configuration

A `LeagueConfig` is *derived* from a real Sleeper league, not hand-written. Fetch the league and convert it:

In [4]:
from nuclearff.config import dump_league_config, league_config_from_sleeper
from nuclearff.sleeper import SleeperClient

league_id = "1367225133634191360"

with SleeperClient() as client:
    league_json = client.get_league(league_id)

league_cfg = league_config_from_sleeper(league_json)
print(league_cfg.name, league_cfg.season, league_cfg.num_teams)
print(league_cfg.roster.counts)

NUCLEARFF REDRAFT 2026 10
{'QB': 1, 'RB': 2, 'WR': 2, 'TE': 1, 'FLEX': 3, 'BN': 6}


`ScoringSettings` and `RosterSlots` both store their data as a generic mapping (`scoring.values`, `roster.counts`) rather than one named field per Sleeper key -- a league can send scoring or roster-slot keys this project has never seen (an IDP league's tackle/sack keys, a superflex slot), and a generic mapping absorbs them instead of failing to parse:

In [5]:
league_cfg.scoring.values["rec"], league_cfg.scoring.values["pass_td"]

(1.0, 6.0)

`playoff_week_start` -- the first week of the fantasy playoffs -- is parsed through the same way, straight from Sleeper's own `settings.playoff_week_start` rather than a hardcoded guess:

In [6]:
league_cfg.playoff_week_start

15

It's `None` for a league that doesn't expose one (a "Chopped" league has no bracket at all) -- `13_league_history.ipynb`'s regular-season/playoff split uses exactly this field rather than assuming a fixed week range.

Save it so it can be reloaded without hitting Sleeper again:

In [7]:
from nuclearff.config import load_league_config

path = dump_league_config(
    league_cfg, cfg.paths.leagues_dir / f"{league_cfg.league_id}.yaml"
)
reloaded = load_league_config(path)
assert reloaded == league_cfg
path

PosixPath('/Users/nmacdonald/projects/sandbox/nuclearff/examples/demo/configs/leagues/1367225133634191360.yaml')

## Replacement rank and starter demand

`LeagueConfig` is where value-based drafting's replacement level comes from (used throughout `08_valuation.ipynb`). `wr_starter_demand` (and the more general `starter_demand` for any position) accounts for locked WR slots *and* the assumed WR share of FLEX slots:

In [8]:
print("WR starter demand:", league_cfg.wr_starter_demand())
print("WR replacement rank (vols):", league_cfg.replacement_rank("WR"))
print("WR replacement rank (vorp):", league_cfg.replacement_rank("WR", baseline="vorp"))
print("RB replacement rank (vols):", league_cfg.replacement_rank("RB"))

WR starter demand: 35.0
WR replacement rank (vols): 35
WR replacement rank (vorp): 53
RB replacement rank (vols): 30


For this league -- 10 teams, 2 WR + 3 FLEX slots, FLEX assumed 50% WR by default -- VOLS (Value Over Last Starter) says the 35th-best WR is replacement level. VORP (Value Over Replacement Player) goes deeper, adding a fraction of the league's bench slots on the theory that a real share of bench spots are speculative WR stashes -- landing at a deeper rank instead. Neither number is "the" replacement level; they're two different, named assumptions about where a bench player becomes replaceable, and `08_valuation.ipynb` uses both.

## What's Next

With a project configuration and a real league's `LeagueConfig` in hand, `03_sleeper_api.ipynb` covers the Sleeper API itself -- everything `league_config_from_sleeper` above reads from, and every other endpoint Sleeper exposes.

## See Also

- `01_introduction.ipynb` -- installation.
- [Configuration](https://nolmacdonald.github.io/nuclearff/tutorial/02_configuration.html) -- the matching docs page.